# Star Intersection Analysis — Gaussian Error Distributions

Analysis of celestial theodolite intersection data comparing FE vs GE model predictions.

**Note:** Observations with an estimated observation distance exceeding 100 km are excluded from this analysis. Distance is estimated from the GE terrestrial drop angle as d ≈ 2R·θ_drop. Currently this excludes North Peak / HD55892 (~131 km).

**How to use this notebook:**
- Run cells in order with `Shift+Enter` (runs current cell and moves to next)
- Or use `Ctrl+Enter` to run a cell and stay on it
- The toolbar has a ▶ button to run the selected cell
- `Kernel > Restart & Run All` re-runs everything from scratch

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

# Inline plots in the notebook
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = '#fafafa'

# Color scheme
FE_COLOR = '#a0e632'       # bright lime green (predicted)
FE_COLOR_MEAS = '#4a8a0e'  # darker lime (measured)
GE_COLOR = '#ff1493'       # hot pink (predicted)
GE_COLOR_MEAS = '#cc1e8a'  # softer deep pink (measured)

R_EARTH_M = 6_371_000  # mean Earth radius in meters
MAX_DIST_M = 100_000   # exclude observations beyond 100 km

def deg2rad(d):
    return d * np.pi / 180.0

def distance_from_drop(drop_dd):
    """Estimate observation distance (m) from GE terrestrial drop angle."""
    return 2.0 * R_EARTH_M * deg2rad(drop_dd)

def sigma_to_meters(sigma_dd, dist_m):
    """Convert angular 1σ (degrees) to linear error (m) at a given distance."""
    return dist_m * deg2rad(sigma_dd)

print('Libraries loaded.')

## Methodology — Gaussian Error Distribution

### Error Model

We assume measurement errors follow a **normal (Gaussian) distribution**:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \, e^{-\frac{(x - \mu)^2}{2\sigma^2}}$$

where:
- $\mu$ = mean of the residuals (systematic bias)
- $\sigma$ = standard deviation (measurement uncertainty)

### Fitting Process

1. **Compute residuals**: For each observation, the residual $\Delta$ is the difference between the measured intersection angle and the model's predicted angle:
$$\Delta_i = \theta_{\text{measured},i} - \theta_{\text{predicted},i}$$

2. **Maximum Likelihood Estimation (MLE)**: We fit a Gaussian to the residuals using `scipy.stats.norm.fit()`, which computes the MLE parameters:
$$\hat{\mu} = \frac{1}{n}\sum_{i=1}^{n} \Delta_i \qquad \hat{\sigma} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(\Delta_i - \hat{\mu})^2}$$

   Note: MLE uses $n$ in the denominator (population std), not $n-1$ (sample std). With $n=17$ observations the difference is small.

3. **Normality validation**: The **Shapiro-Wilk test** checks whether the residuals are consistent with a normal distribution ($H_0$: data is normal). A p-value > 0.05 means we cannot reject normality.

### Error Bars (1$\sigma$)

The error bars on measured values represent $\pm 1\sigma$ — one standard deviation from the fitted Gaussian. Under a normal distribution:
- **68.3%** of measurements fall within $\pm 1\sigma$
- **95.4%** fall within $\pm 2\sigma$
- **99.7%** fall within $\pm 3\sigma$

### Angular-to-Linear Conversion

To express error bars in meters, we estimate the observation distance from the GE terrestrial drop angle and convert:

$$d \approx 2 R_\oplus \cdot \theta_{\text{drop}} \qquad \text{(radians)}$$

$$\epsilon_{\text{meters}} = d \cdot \sigma_{\text{rad}}$$

where $R_\oplus = 6{,}371{,}000$ m is Earth's mean radius.

### Pass/Fail Criteria

For each observation, we compute how many standard deviations the predicted value is from the measured value:

$$n_\sigma = \frac{|\theta_{\text{predicted}} - \theta_{\text{measured}}|}{\sigma}$$

| Result | Condition | Interpretation |
|--------|-----------|----------------|
| **WITHIN 1$\sigma$** (green) | $n_\sigma \leq 1.0$ | Prediction consistent with measurement |
| **WITHIN 2$\sigma$** (amber) | $1.0 < n_\sigma \leq 2.0$ | Marginal — possible systematic error |
| **OUTSIDE 2$\sigma$** (red) | $n_\sigma > 2.0$ | Prediction inconsistent with measurement |

## 1. Load the Data

In [ ]:
csv_path = Path('data') / 'intersections.csv'

df = pd.read_csv(csv_path, skipinitialspace=True)
df.columns = df.columns.str.strip()

# Estimate observation distance and exclude >100 km
df['dist_m'] = 2.0 * R_EARTH_M * deg2rad(df['GE_terrestrial_drop_dd'])
excluded = df[df['dist_m'] > MAX_DIST_M]
if len(excluded) > 0:
    print('Excluded (>100 km):')
    for _, row in excluded.iterrows():
        print(f"  {row['Peak']} / {row['Star']} (~{row['dist_m']/1000:.0f} km)")
df = df[df['dist_m'] <= MAX_DIST_M].reset_index(drop=True)

df['label'] = df['Peak'] + ' / ' + df['Star']
print(f'\nLoaded {len(df)} observations (after exclusions)')
df

## 2. Summary Statistics + Normality Test

In [ ]:
fe_vals = df['ΔFE_intersection_dd'].values
ge_vals = df['ΔGE_intersection__dd'].values

fe_mu, fe_sigma = stats.norm.fit(fe_vals)
ge_mu, ge_sigma = stats.norm.fit(ge_vals)

summary = pd.DataFrame({
    'Metric': ['n', 'Mean (μ)', 'Std Dev (σ)', 'Min', 'Max', '|μ| (bias)'],
    'FE Residuals': [len(fe_vals), f'{fe_mu:.4f}°', f'{fe_sigma:.4f}°',
                     f'{fe_vals.min():.4f}°', f'{fe_vals.max():.4f}°', f'{abs(fe_mu):.4f}°'],
    'GE Residuals': [len(ge_vals), f'{ge_mu:.4f}°', f'{ge_sigma:.4f}°',
                     f'{ge_vals.min():.4f}°', f'{ge_vals.max():.4f}°', f'{abs(ge_mu):.4f}°'],
})
print(summary.to_string(index=False))

print('\nShapiro-Wilk normality test:')
for vals, name in [(fe_vals, 'FE'), (ge_vals, 'GE')]:
    stat, p = stats.shapiro(vals)
    verdict = 'normal' if p > 0.05 else 'non-normal'
    print(f'  {name}: W={stat:.4f}, p={p:.4f} ({verdict})')

## 3. Plot 1 — Residual Comparison Bar Chart

Each observation's ΔFE and ΔGE side-by-side, with the population 1σ shown as error bars.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
n = len(df)
x = np.arange(n)
width = 0.38

ax.bar(x - width/2, fe_vals, width, label=f'ΔFE (μ={fe_mu:.3f}°, σ={fe_sigma:.3f}°)',
       color=FE_COLOR, alpha=0.85, yerr=fe_sigma, capsize=3, error_kw={'lw': 0.8})
ax.bar(x + width/2, ge_vals, width, label=f'ΔGE (μ={ge_mu:.3f}°, σ={ge_sigma:.3f}°)',
       color=GE_COLOR, alpha=0.85, yerr=ge_sigma, capsize=3, error_kw={'lw': 0.8})

ax.axhline(0, color='grey', lw=0.6, ls='--')
ax.axhline(fe_mu, color=FE_COLOR_MEAS, lw=0.8, ls=':')
ax.axhline(ge_mu, color=GE_COLOR_MEAS, lw=0.8, ls=':')

ax.set_xticks(x)
ax.set_xticklabels(df['label'], fontsize=7, rotation=45, ha='right')
ax.set_ylabel('Δ Intersection (°)')
ax.set_title('Residuals per Observation — FE vs GE Model')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

## 4. Plot 2 — Histogram + Gaussian Overlay

Distribution of residuals for each model with the best-fit normal curve.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for ax, vals, color, color_dark, model, mu, sigma in [
    (ax1, fe_vals, FE_COLOR, FE_COLOR_MEAS, 'Flat Earth', fe_mu, fe_sigma),
    (ax2, ge_vals, GE_COLOR, GE_COLOR_MEAS, 'Globe Earth', ge_mu, ge_sigma),
]:
    n_bins = max(6, len(vals) // 2)
    ax.hist(vals, bins=n_bins, density=True, alpha=0.6, color=color, edgecolor='white')
    
    x_range = np.linspace(vals.min() - 2*sigma, vals.max() + 2*sigma, 200)
    ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma),
            color=color_dark, lw=2, label=f'N({mu:.3f}, {sigma:.3f}²)')
    
    ax.axvline(mu, color='black', lw=1, ls='--', label=f'μ = {mu:.3f}°')
    ax.axvspan(mu - sigma, mu + sigma, alpha=0.12, color=color, label='±1σ')
    
    ax.set_xlabel('Δ Intersection (°)')
    ax.set_ylabel('Density')
    ax.set_title(f'{model} Residual Distribution')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 5. Plot 3 — Predicted Angle Scatter with Error Bars

FE and GE predicted angles for each observation, with 1σ Gaussian error bars.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(df))

ax.errorbar(x - 0.15, df['FE_pred_angle_dd'], yerr=fe_sigma, fmt='o',
            color=FE_COLOR_MEAS, markersize=5, capsize=3,
            label=f'FE predicted (±{fe_sigma:.3f}°)')
ax.errorbar(x + 0.15, df['GE_pred_angle_dd'], yerr=ge_sigma, fmt='s',
            color=GE_COLOR_MEAS, markersize=5, capsize=3,
            label=f'GE predicted (±{ge_sigma:.3f}°)')

ax.set_xticks(x)
ax.set_xticklabels(df['label'], fontsize=7, rotation=45, ha='right')
ax.set_ylabel('Predicted Angle (°)')
ax.set_title('FE vs GE Predicted Angles with 1σ Error Bars')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 6. Plot 4 — Violin + Swarm Plot

Shows the full distribution shape of residuals for each model, with individual data points overlaid.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

parts = ax.violinplot([fe_vals, ge_vals], positions=[1, 2],
                      showmeans=True, showmedians=True, showextrema=True)

for pc, color in zip(parts['bodies'], [FE_COLOR, GE_COLOR]):
    pc.set_facecolor(color)
    pc.set_alpha(0.4)
for key in ('cmeans', 'cmedians', 'cbars', 'cmins', 'cmaxes'):
    parts[key].set_color('black')

# Overlay individual points
rng = np.random.default_rng(42)
for pos, vals, color in [(1, fe_vals, FE_COLOR_MEAS), (2, ge_vals, GE_COLOR_MEAS)]:
    jitter = rng.uniform(-0.06, 0.06, len(vals))
    ax.scatter(np.full_like(vals, pos) + jitter, vals,
               color=color, alpha=0.7, s=30, zorder=5,
               edgecolors='white', linewidth=0.5)

ax.set_xticks([1, 2])
ax.set_xticklabels(['ΔFE (Flat Earth)', 'ΔGE (Globe Earth)'])
ax.set_ylabel('Δ Intersection (°)')
ax.set_title('Residual Distribution — Violin + Swarm')
ax.axhline(0, color='grey', lw=0.6, ls='--')
ax.grid(axis='y', alpha=0.3)

# Stats annotations
for pos, mu, sigma, color in [(1, fe_mu, fe_sigma, FE_COLOR_MEAS),
                                (2, ge_mu, ge_sigma, GE_COLOR_MEAS)]:
    ax.text(pos, ax.get_ylim()[1] * 0.92,
            f'μ={mu:.3f}°\nσ={sigma:.3f}°',
            ha='center', fontsize=9, color=color,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=color, alpha=0.8))

fig.tight_layout()
plt.show()

## 7. Paired Model Comparison — Which Model Is Consistently Closer?

The previous plots characterize each model's residuals independently. To determine which model **consistently** makes the closer prediction, we compare them observation-by-observation using absolute residuals: |ΔFE| vs |ΔGE|.

**Wilcoxon Signed-Rank Test**: A non-parametric paired test that asks whether one model's absolute residuals are systematically smaller. Unlike a t-test, it makes no normality assumption about the differences.

- $H_0$: The distribution of |ΔFE| − |ΔGE| is symmetric about zero
- If $p < 0.05$: reject $H_0$ — one model is significantly and consistently closer

In [ ]:
# Win/loss tally + Wilcoxon signed-rank test
fe_abs = df['ΔFE_intersection_dd'].abs().values
ge_abs = df['ΔGE_intersection__dd'].abs().values
diff = fe_abs - ge_abs  # negative = FE closer

fe_wins = int(np.sum(diff < 0))
ge_wins = int(np.sum(diff > 0))
ties = int(np.sum(diff == 0))

print(f'Paired comparison (|ΔFE| vs |ΔGE| per observation):')
print(f'  FE closer: {fe_wins}/{len(diff)}')
print(f'  GE closer: {ge_wins}/{len(diff)}')
if ties:
    print(f'  Tied:      {ties}/{len(diff)}')

w_stat, w_p = stats.wilcoxon(fe_abs, ge_abs, alternative='two-sided')
print(f'\nWilcoxon signed-rank test:')
print(f'  W = {w_stat:.1f}, p = {w_p:.4f}')
if w_p < 0.05:
    winner = 'FE' if np.median(diff) < 0 else 'GE'
    print(f'  Significant (p < 0.05): {winner} residuals are systematically smaller')
else:
    print(f'  Not significant (p >= 0.05): no systematic difference')

### Plot 5 — Paired Difference Bar Chart

For each observation: |ΔFE| − |ΔGE|. Bars extending left = FE was closer.

In [ ]:
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(14, 6))
y = np.arange(len(df))
colors = [FE_COLOR if d < 0 else GE_COLOR for d in diff]

ax.barh(y, diff, color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='grey', lw=0.8, ls='--')

tally = f'FE closer: {fe_wins}    GE closer: {ge_wins}'
if ties:
    tally += f'    Tied: {ties}'
ax.set_title(f'Which Model Was Closer?  —  {tally}')
ax.set_yticks(y)
ax.set_yticklabels(df['label'], fontsize=7)
ax.set_xlabel('|ΔFE| − |ΔGE|  (°)     ← FE closer | GE closer →')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()
ax.legend(handles=[Patch(facecolor=FE_COLOR, label='FE closer'),
                   Patch(facecolor=GE_COLOR, label='GE closer')],
          fontsize=8, loc='lower right')
fig.tight_layout()
plt.show()

### Plot 6 — |ΔFE| vs |ΔGE| Scatter

Each point is one observation. Points above the y = x diagonal indicate FE had the smaller error.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(fe_abs, ge_abs, s=60, zorder=5,
           color='#555555', edgecolors='white', linewidth=0.8)

for i, row in df.iterrows():
    ax.annotate(row['Peak'], (fe_abs[i], ge_abs[i]),
                textcoords='offset points', xytext=(5, 5),
                fontsize=6, color='#333333')

lo = 0
hi = max(fe_abs.max(), ge_abs.max()) * 1.15
ax.plot([lo, hi], [lo, hi], ls='--', lw=1, color='grey', zorder=1)

# Above y=x: |ΔGE| > |ΔFE| → FE closer
ax.fill_between([lo, hi], [lo, hi], hi, alpha=0.06, color=FE_COLOR,
                label='FE closer (above line)')
ax.fill_between([lo, hi], lo, [lo, hi], alpha=0.06, color=GE_COLOR,
                label='GE closer (below line)')

ax.set_xlabel('|ΔFE| (°)')
ax.set_ylabel('|ΔGE| (°)')
ax.set_title('|ΔFE| vs |ΔGE| — Points Above Diagonal = FE Closer')
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect('equal')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### Plot 7 — Empirical CDF of Absolute Residuals

The curve that rises faster (further left) has consistently smaller errors. If one curve fully dominates the other, that model has **first-order stochastic dominance**.

In [ ]:
import matplotlib.ticker as ticker

fig, ax = plt.subplots(figsize=(10, 7))

fe_sorted = np.sort(fe_abs)
ge_sorted = np.sort(ge_abs)
n = len(fe_sorted)
ecdf = np.arange(1, n + 1) / n

ax.step(fe_sorted, ecdf, where='post', color=FE_COLOR, lw=2, label='FE |Δ|')
ax.step(ge_sorted, ecdf, where='post', color=GE_COLOR, lw=2, label='GE |Δ|')

# Shade area between curves
all_x = np.sort(np.unique(np.concatenate([fe_sorted, ge_sorted])))
fe_ecdf_interp = np.searchsorted(fe_sorted, all_x, side='right') / n
ge_ecdf_interp = np.searchsorted(ge_sorted, all_x, side='right') / n
ax.fill_betweenx(
    np.linspace(0, 1, 100),
    np.interp(np.linspace(0, 1, 100), fe_ecdf_interp, all_x),
    np.interp(np.linspace(0, 1, 100), ge_ecdf_interp, all_x),
    alpha=0.08, color='grey')

ax.set_xlabel('|Δ Intersection| (°)')
ax.set_ylabel('Cumulative Probability')
ax.set_title('Empirical CDF of Absolute Residuals — Leftward = Better')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.02)
ax.set_xlim(0, None)
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
fig.tight_layout()
plt.show()

## 8. Individual Peak Plots — Predicted vs Measured

Each peak gets its own two-panel chart (FE left, GE right) comparing predicted vs measured angles.
Error bars are on the measured values. A badge shows whether the prediction falls within 1σ, 2σ, or outside 2σ.

**Note:** All peaks are plotted (including those excluded from the Gaussian fit). The σ used for error bars comes from the filtered dataset only (observations ≤100 km).

In [ ]:
# Reload full dataset (including excluded peaks) for individual plots
df_all = pd.read_csv(csv_path, skipinitialspace=True)
df_all.columns = df_all.columns.str.strip()
df_all['dist_m'] = 2.0 * R_EARTH_M * deg2rad(df_all['GE_terrestrial_drop_dd'])

# σ comes from the filtered data (already computed above as fe_sigma, ge_sigma)
peaks_all = df_all.groupby('Peak', sort=False)

for peak_name, peak_df in peaks_all:
    n = len(peak_df)
    stars = peak_df['Star'].values
    fe_pred = peak_df['FE_pred_angle_dd'].values
    ge_pred = peak_df['GE_pred_angle_dd'].values
    fe_delta = peak_df['ΔFE_intersection_dd'].values
    ge_delta = peak_df['ΔGE_intersection__dd'].values
    fe_meas = fe_pred + fe_delta
    ge_meas = ge_pred + ge_delta

    fig, (ax_fe, ax_ge) = plt.subplots(1, 2, figsize=(12, 6), facecolor='#fafafa')
    fig.suptitle(peak_name, fontsize=14, fontweight='bold', y=0.98)

    for ax, pred, meas, delta, sigma, color, color_m, label in [
        (ax_fe, fe_pred, fe_meas, fe_delta, fe_sigma, FE_COLOR, FE_COLOR_MEAS, 'Flat Earth'),
        (ax_ge, ge_pred, ge_meas, ge_delta, ge_sigma, GE_COLOR, GE_COLOR_MEAS, 'Globe Earth'),
    ]:
        ax.set_facecolor('#fafafa')
        x = np.arange(n)
        width = 0.3

        ax.bar(x - width/2, pred, width, color=color, alpha=0.8, label=f'{label} predicted')
        ax.bar(x + width/2, meas, width, color=color_m, alpha=0.8,
               label=f'{label} measured', yerr=sigma, capsize=5,
               error_kw={'lw': 1.2, 'capthick': 1.2})

        all_vals = np.concatenate([pred, meas + sigma, meas - sigma])
        data_top = all_vals.max()
        data_bottom = all_vals.min()
        data_span = max(data_top - data_bottom, 0.3)
        ax.set_ylim(max(0, data_bottom - data_span * 0.15), data_top + data_span * 0.70)
        text_offset = data_span * 0.04

        for i in range(n):
            drop_dd = peak_df['GE_terrestrial_drop_dd'].values[i]
            dist_m = distance_from_drop(drop_dd)
            err_m = sigma_to_meters(sigma, dist_m)
            n_sigma = abs(pred[i] - meas[i]) / sigma

            if n_sigma <= 1.0:
                verdict = f'WITHIN 1σ ({n_sigma:.1f}σ)'
                v_color = '#27ae60'
            elif n_sigma <= 2.0:
                verdict = f'WITHIN 2σ ({n_sigma:.1f}σ)'
                v_color = '#f39c12'
            else:
                verdict = f'OUTSIDE 2σ ({n_sigma:.1f}σ)'
                v_color = '#c0392b'

            y_label = max(pred[i], meas[i] + sigma) + text_offset
            ax.text(x[i], y_label, f'Δ={delta[i]:+.2f}° (±{err_m:.0f} m)',
                    ha='center', va='bottom', fontsize=8, color=color_m, fontweight='bold')
            y_badge = y_label + data_span * 0.12
            ax.text(x[i], y_badge, verdict, ha='center', va='bottom',
                    fontsize=8, fontweight='bold', color='white',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=v_color,
                              edgecolor='none', alpha=0.9))

        for i in range(n):
            drop = peak_df['GE_terrestrial_drop_dd'].values[i]
            ax.text(x[i], min(pred[i], meas[i]) - text_offset, f'drop = {drop:.2f}°',
                    ha='center', va='top', fontsize=8, color='black', fontweight='bold')

        ax.set_xticks(x)
        ax.set_xticklabels(stars, fontsize=9)
        ax.set_ylabel('Angle (°)')
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(axis='y', alpha=0.3)

    fig.tight_layout()
    plt.show()

## 9. Save All Plots

Run this cell to regenerate and save all PNGs (overview + individual) to the `plots/` directory.

In [ ]:
import subprocess, os
# Run both scripts to regenerate all PNGs
for script in ['plot_intersections.py', 'plot_individuals.py']:
    result = subprocess.run(
        ['python3', script],
        capture_output=True, text=True,
        env={**os.environ, 'MPLBACKEND': 'Agg'}
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f'ERROR in {script}:', result.stderr)